# Homework — Math Score vs Parental level of education

Dataset: [Students Performance in Exams](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams)

Goal: the dataset has many columns, but per the assignment we focus on just two: predict `math score` from
`parental level of education` with a simple linear regression, and report how accurate that is.

Task checklist: **A)** clean data · **B)** build model · **C)** measure accuracy · **D)** document (this notebook).

## A. Load & Clean the Data

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pd.set_option("display.precision", 2)

In [2]:
raw = pd.read_csv("../data/students_performance.csv")
raw.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [3]:
# Check for missing values / duplicate rows on the FULL row first -- checking only our two
# columns of interest would flag unrelated students who just happen to share a gender + score
# as "duplicates", which is wrong.
print("shape:", raw.shape)
print("missing values:\n", raw.isna().sum().sum(), "total")
print("duplicate rows (full record):", raw.duplicated().sum())

shape: (1000, 8)
missing values:
 0 total
duplicate rows (full record): 0


In [12]:
df = raw.drop_duplicates()[["parental level of education", "math score"]].copy()
df["edu_code"] = df["parental level of education"].map({"some high school": 0, "high school": 1, "some college": 2, "associate's degree": 3, "bachelor's degree": 4, "master's degree": 5})

print("edu values:", df["edu_code"].unique())
df.describe(include="all")

edu values: [4 2 5 3 1 0]


,parental level of education,math score,edu_code
count,1000,1000.00,1000.00
unique,6,NaN,NaN
top,some college,NaN,NaN
freq,226,NaN,NaN
mean,NaN,66.09,2.08
std,NaN,15.16,1.46
min,NaN,0.00,0.00
25%,NaN,57.00,1.00
50%,NaN,66.00,2.00
75%,NaN,77.00,3.00


No missing values and no duplicate student records. The only real cleaning step is encoding `gender` as a 0/1 number (`gender_code`), since linear regression needs numeric input.

## Quick Look at the Data

In [5]:
box_figure = px.box(df, x="gender", y="math score", color="gender",
                     title="Math Score Distribution by Gender", template="plotly_white",
                     color_discrete_map={"male": "#2563eb", "female": "#dc2626"})
box_figure.update_layout(width=650, height=450, showlegend=False)
box_figure.show()

The two distributions overlap heavily with only a modest gap in the medians — a hint that gender alone won't explain much of the spread in math scores.

## B. Build the Linear Regression Model

In [6]:
X = df[["gender_code"]]
y = df["math score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"coefficient (gender_code): {model.coef_[0]:.2f}")
print(f"intercept (male's predicted score): {model.intercept_:.2f}")
print(f"predicted female score: {model.intercept_ + model.coef_[0]:.2f}")

coefficient (gender_code): -4.59
intercept (male's predicted score): 68.91
predicted female score: 64.32


With a single 0/1 feature, the fitted line only ever produces **two** possible predictions: the intercept for males, and intercept + coefficient for females. In other words, the model has learned the average math score of each group.

## C. Measure the Model's Accuracy

In [7]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print(f"R^2:  {r2:.3f}   (fraction of variance in math score explained by gender)")
print(f"MAE:  {mae:.2f}  points   (average absolute prediction error)")
print(f"RMSE: {rmse:.2f}  points   (typical prediction error, penalizes big misses more)")

R^2:  0.028   (fraction of variance in math score explained by gender)
MAE:  12.05  points   (average absolute prediction error)
RMSE: 15.38  points   (typical prediction error, penalizes big misses more)


In [8]:
compare_figure = go.Figure()
compare_figure.add_trace(go.Scatter(x=df.loc[X_test.index, "gender"], y=y_test, mode="markers",
                                     marker=dict(size=8, color="#94a3b8"), name="actual"))
compare_figure.add_trace(go.Scatter(x=["male", "female"],
                                     y=[model.intercept_, model.intercept_ + model.coef_[0]],
                                     mode="markers", marker=dict(size=16, color="#dc2626", symbol="x"),
                                     name="predicted (group mean)"))
compare_figure.update_layout(title="Actual Scores vs the Model's Two Predictions", xaxis_title="Gender",
                              yaxis_title="Math score", template="plotly_white", width=650, height=450)
compare_figure.show()

## D. Summary

- **Data**: `gender` and `math score` only, as scoped by the assignment. No missing values; a few duplicate
  rows dropped. `gender` was encoded as `gender_code` (male=0, female=1) since linear regression needs numbers.
- **Model**: linear regression with one binary feature — equivalent to predicting each gender's average
  math score, trained on an 80/20 split.
- **Accuracy**: R² is low (see printed metric above) — a single yes/no feature like gender explains only a
  small slice of the variance in math scores; MAE/RMSE stay close to the overall score spread.
- **Takeaway**: this is an honest example of a *weak* linear predictor. The lesson isn't "linear regression
  failed" — it's that predictive power comes from the feature, not the algorithm. Score is driven mostly by
  factors outside gender (e.g. test preparation, parental education) that this restricted model doesn't see.